# Full Paper → Full UTD19 → Berlin → SUMO End-to-End Tutor

এই notebook uploaded VTC 2024 paper-কে **concept + code + input/output + experiment + simulation** হিসেবে শেখায়।

Pipeline:

```text
Official full UTD19
→ correct unit-aware preprocessing
→ five ML models
→ Random Forest Berlin prediction
→ actual Berlin comparison
→ current Berlin OSM → SUMO
→ audited detector mapping
→ randomTrips + TAZ + duarouter
→ routeSampler optimize=full
→ 24-hour headless SUMO
→ GEH + plots
→ Mac SUMO-GUI bundle
```

> এটি authors-এর unpublished code নয়। Exact preprocessing, feature table, split,
> OSM snapshot, detector mapping ও commands paper প্রকাশ করেনি। প্রতিটি assumption
> project-এ record করা হয়।

## Before running

- Full UTD19 raw CSV প্রায় 6.65 GB।
- Full Berlin experiment বহু ঘণ্টা নিতে পারে।
- Colab `/content` temporary। Final workspace Drive-এ archive করবে।
- প্রথমে demo mode end-to-end certify করা strongly recommended, তারপর subset, তারপর full।

In [ ]:
!df -h /content
import os, platform
print(platform.platform())

# Setup 1 — Upload project ZIP

Colab-এর left **Files** panel-এ upload করো:

`utd19_berlin_sumo_e2e_v3.zip`

In [ ]:
ZIP_PATH = "/content/utd19_berlin_sumo_e2e_v3.zip"
!test -f "$ZIP_PATH" && echo "ZIP found" || echo "Upload the ZIP first"
!unzip -q -o "$ZIP_PATH" -d /content
%cd /content/utd19_berlin_sumo_e2e_v3
!find . -maxdepth 2 -type f | sort | head -80

# Setup 2 — Install official tools and Python packages

Installs `osmium`, SUMO binaries/tools, DuckDB, scikit-learn, mapping and plotting libraries.

In [ ]:
!bash scripts/install_colab.sh

In [ ]:
!sumo --version | head -4
!duarouter --version | head -4
!netconvert --version | head -4
!python -m src.pipeline --config config/demo.yaml --list-steps

# Paper page viewer

Rendered pages are bundled so every table/figure can be read beside its code stage.

In [ ]:
from IPython.display import Image, display

def show_page(number):
    display(Image(filename=f"paper/pages/page_{number}.png", width=900))

show_page(1)

# Lesson 1 — Page 1: problem and contribution

SUMO can simulate individual vehicles, but realistic large-city demand is difficult.
Paper proposes learning traffic count patterns from real multi-city data and then
calibrating SUMO routes to the predicted counts.

**Key distinction:** ML predicts flow/count; it does not predict each complete route.

# Lesson 2 — Page 2: related scenarios and methodology

Bologna/Austria/Turin use real or proprietary data; Cologne/Monaco/Luxembourg use
synthetic population/activity models. The paper targets faster city-general demand generation.

In [ ]:
show_page(2)

# Lesson 3 — Pages 2–3: UTD19 dataset and Table II

Raw measurements:

```text
city, detid, day, interval, flow, occ, speed, error
```

Critical official-manual unit:

```text
flow = vehicles/hour/lane
```

Although observations occur every 3–5 minutes, the values are rates. Therefore the
hourly reconstruction uses average/median within an hour—not the sum of rates.

In [ ]:
show_page(3)

# Lesson 4 — Pages 3–4: Berlin external verification

Berlin provides hourly `Nveh`, speed, passenger/truck composition and quality.
It is external verification, not UTD19 training data.

In [ ]:
show_page(4)

# Lesson 5 — Page 5: routeSampler calibration and model results

Paper compares Linear Regression, Linear SVM, Polynomial Regression, Decision Tree,
and Random Forest. Random Forest has the lowest reported SMAPE/MAE.

RouteSampler receives candidate routes plus predicted edge counts and uses
`optimize=full` in the paper.

In [ ]:
show_page(5)
import pandas as pd
paper_results = pd.DataFrame([
    ["Linear Regression",72.08,194.49],
    ["Linear SVM",71.53,187.16],
    ["Polynomial Regression",69.51,172.34],
    ["Decision Tree",19.74,79.85],
    ["Random Forest",17.12,71.04],
], columns=["Model","Paper SMAPE %","Paper MAE"])
display(paper_results)

# Lesson 6 — Page 6: actual/predicted plots and GEH

Paper plots Berlin actual counts for 02–06 February 2021 against one predicted day.

```text
GEH = sqrt(2 × (M−C)^2 / (M+C))
```

The project reports both GEH and absolute mismatch because the abstract and body use
slightly inconsistent wording.

In [ ]:
show_page(6)
from src.metrics import geh
for predicted, simulated in [(100,100),(100,110),(100,150),(1000,1100)]:
    print(predicted, simulated, float(geh([predicted],[simulated])[0]))

# Lesson 7 — Page 7: conclusion, 24-hour trajectory and limitations

Paper reports ~891.7 km², >260 detectors, >1.2M vehicles and 24 hours. Those are
paper reference values. This reconstruction must independently measure its own area,
detectors, vehicles, GEH, waiting and teleports.

In [ ]:
show_page(7)

# Stage A — Run the complete demo first

Demo uses synthetic UTD19-shaped data and a small grid network, but executes the same
code path through routeSampler, SUMO, GEH and GUI packaging.

In [ ]:
!python -m src.pipeline --config config/demo.yaml --accept-auto-mapping

## Inspect demo evidence

In [ ]:
from pathlib import Path
import json, pandas as pd
DEMO = Path('/content/utd19_berlin_e2e_demo')
display(pd.read_csv(DEMO/'models/model_comparison.csv'))
print(json.loads((DEMO/'reports/final_evaluation.json').read_text()))
print("GUI ZIP:", DEMO/'gui_bundle.zip')

# Stage B — Choose official subset or full mode

- `subset`: full official UTD19 training, 30 Berlin detectors, clipped road network.
- `full`: full official UTD19, all matched Berlin detectors, full Berlin network, 24 hours.

The full mode is selected below because that is the target experiment.

In [ ]:
MODE = "full"   # change to "subset" only for certification
CONFIG = f"config/{MODE}.yaml"
WORKSPACE = Path(f"/content/utd19_berlin_e2e_{MODE}")
print(CONFIG, WORKSPACE)

# Stage C — Official full UTD19 → preprocessing → five models

This downloads the official ETH files, scans every raw row with DuckDB, creates the
hourly model table and trains all five model families.

This is the largest ML stage and may take hours.

In [ ]:
!python -m src.pipeline --config "$CONFIG" --to-step 03_train_models

## Verify full-data audit and our model results

In [ ]:
import json, pandas as pd
print(json.loads((WORKSPACE/'processed/utd19/utd19_audit.json').read_text()))
display(pd.read_csv(WORKSPACE/'models/model_comparison.csv'))
display(pd.read_csv(WORKSPACE/'models/paper_vs_reimplementation.csv'))

# Stage D — Official Berlin archive → actual data → current Berlin OSM network

The official archive is dynamically discovered from the Berlin public container.
Current historical files may differ from the paper snapshot because the archive was revised.

In [ ]:
!python -m src.pipeline   --config "$CONFIG"   --from-step 04_download_berlin   --to-step 07_build_network

## Inspect Berlin and network audits

In [ ]:
print(json.loads((WORKSPACE/'processed/berlin/berlin_audit.json').read_text()))
print(json.loads((WORKSPACE/'sumo/network/network_manifest.json').read_text()))

# Stage E — Detector-to-edge mapping: mandatory correctness gate

Run mapping generation. It creates a candidate CSV and interactive HTML. Full mode
stops until an accepted mapping exists.

In [ ]:
# This cell is expected to stop at the manual mapping gate on the first run.
!python -m src.pipeline --config "$CONFIG" --from-step 08_map_detectors --to-step 08_map_detectors

## Open the mapping review

Use Colab Files panel to download/open:

```text
<workspace>/sumo/mapping/mapping_review.html
```

Review direction, divided road, bridge/tunnel and lane count. Then copy/edit
`mapping_review_template.csv` as `mapping_accepted.csv`.

For a fully automated but **unreviewed** run, execute the next cell.

In [ ]:
AUTO_ACCEPT_UNREVIEWED_MAPPING = False
if AUTO_ACCEPT_UNREVIEWED_MAPPING:
    !python -m src.pipeline       --config "$CONFIG"       --from-step 08_map_detectors       --to-step 08_map_detectors       --force       --accept-auto-mapping
else:
    print("Keep False for academic-quality manual review.")

# Stage F — Prediction → TAZ/random candidates → routeSampler → 24-hour SUMO

After `mapping_accepted.csv` exists, this resumes all remaining stages:

- Berlin feature alignment and Random Forest prediction;
- randomTrips + duarouter;
- grid TAZ trips + duarouter;
- detector-targeted coverage routes;
- 100% candidate coverage gate;
- routeSampler `optimize=full`;
- 24-hour headless SUMO;
- simulated count extraction;
- GEH/plots;
- GUI bundle.

In [ ]:
!python -m src.pipeline   --config "$CONFIG"   --from-step 09_build_berlin_features   --accept-auto-mapping

# Stage G — Inspect all final inputs and outputs

In [ ]:
for path in [
    WORKSPACE/'processed/berlin/berlin_predictions.csv',
    WORKSPACE/'sumo/candidates/candidate_coverage.csv',
    WORKSPACE/'sumo/calibration/edge_predictions.csv',
    WORKSPACE/'reports/geh_summary.csv',
    WORKSPACE/'reports/actual_prediction_summary.csv',
]:
    print('
', path)
    if path.exists():
        display(pd.read_csv(path).head(20))
    else:
        print('missing')

## Display generated experiment figures

In [ ]:
from IPython.display import Image, display
figure_paths = sorted((WORKSPACE/'reports').glob('*.png'))
print('figures:', len(figure_paths))
for path in figure_paths[:12]:
    print(path.name)
    display(Image(filename=str(path), width=850))

# Stage H — Download and view live moving vehicles on Mac

Colab runs the numerical simulation headlessly. Download:

```text
<workspace>/gui_bundle.zip
```

On your Mac:

```bash
unzip gui_bundle.zip
cd gui_bundle
chmod +x run_gui.sh
./run_gui.sh
```

In [ ]:
print('GUI bundle:', WORKSPACE/'gui_bundle.zip')
!ls -lh "$WORKSPACE/gui_bundle.zip" 2>/dev/null || true

# Stage I — Save the full experiment to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!python -m src.archive_workspace   --workspace "$WORKSPACE"   --destination "/content/drive/MyDrive/{WORKSPACE.name}_backup"

# Full-paper mastery questions

1. ML target কী এবং complete vehicle route কে তৈরি করে?
2. UTD19 `flow` unit কী? Hourly sum কেন ভুল?
3. Berlin কেন external verification?
4. Target per-lane থেকে detector-total conversion কীভাবে করা হয়েছে?
5. Paper-এর exact model numbers আমাদের result-এর সঙ্গে একই হওয়া জরুরি নয় কেন?
6. OSM detector nearest edge automatically accept করা risky কেন?
7. randomTrips alone কেন insufficient?
8. TAZ candidate এবং targeted coverage route-এর ভূমিকা কী?
9. RouteSampler-এর input তিনটি category কী?
10. `optimize=full` কীভাবে paper-এর সঙ্গে align করে?
11. GEH এবং absolute mismatch-এর পার্থক্য কী?
12. Predicted edge entries এবং simultaneously running vehicles কেন time-shifted?
13. Full project complete বলার আগে কোন evidence files দেখতে হবে?

# Final status rule

Only this evidence-backed statement is valid:

> We completed a paper-faithful end-to-end reconstruction using official public
> datasets and current official SUMO tools, while documenting every assumption
> required because the authors’ exact implementation was not published.

Do not claim an exact authors-code reproduction or exact 1.2M-vehicle replication
unless the generated manifests and outputs independently establish it.